# Task 3

## 1. Install and start Ollama

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (4,684 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
import subprocess, time, os, requests, json, statistics, pandas as pd
from IPython.display import display

ollama_log = open("/tmp/ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
OLLAMA_URL = "http://127.0.0.1:11434"

for attempt in range(40):
    try:
        r = requests.get(OLLAMA_URL + "/api/tags", timeout=2)
        if r.status_code == 200:
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start. Inspect /tmp/ollama.log")

print("Ollama is running:", requests.get(OLLAMA_URL + "/api/version").json())


Ollama is running: {'version': '0.40.16'}


## 2. Download two quantised versions

Both tags are Qwen2.5 1.5B Instruct, with Q4_K_M and Q8_0 quantisation respectively. Model tags can be checked in the [official Ollama Qwen2.5 library](https://ollama.com/library/qwen2.5/tags).


In [6]:
MODEL_Q4 = "qwen2.5:1.5b-instruct-q4_K_M"
MODEL_Q8 = "qwen2.5:1.5b-instruct-q8_0"

!ollama pull qwen2.5:1.5b-instruct-q4_K_M
!ollama pull qwen2.5:1.5b-instruct-q8_0

print("Both model variants downloaded.")




Both model variants downloaded.


## 3. Ten fixed evaluation prompts

In [7]:
prompts = [
    "Explain the difference between artificial intelligence, machine learning, and deep learning in simple terms.",
    "Write a Python function that checks whether a number is prime. Explain how it works.",
    "Summarise the purpose of the NIST AI Risk Management Framework in three bullet points.",
    "A shop sells 3 notebooks for 90 rupees. What is the price of 7 notebooks at the same rate? Show the calculation.",
    "Explain overfitting in machine learning and give two ways to reduce it.",
    "Write a polite email asking a professor for an extension on an assignment.",
    "What is the difference between RAM and storage in a computer?",
    "Give a step-by-step plan for cleaning and preparing a dataset before training a machine learning model.",
    "Explain quantisation in large language models to a beginner.",
    "A server receives 120 requests in 30 seconds. Calculate the average requests per second and explain."
]
assert len(prompts) == 10
print("Prompts:", len(prompts))


Prompts: 10


## 4. Benchmark helpers

Ollama reports generated token count (`eval_count`) and generation duration (`eval_duration`, nanoseconds). Calculate tokens/second as `eval_count / (eval_duration / 1e9)`. Memory readings are snapshots, not guaranteed peak values.


In [8]:
def gpu_memory():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True
        ).strip().splitlines()[0]
        used, total = [int(x.strip()) for x in out.split(",")]
        return {"used_mb": used, "total_mb": total}
    except Exception:
        return {"used_mb": None, "total_mb": None}

def loaded_models():
    try:
        return requests.get(OLLAMA_URL + "/api/ps", timeout=10).json().get("models", [])
    except Exception:
        return []

def generate_one(model_name, prompt):
    payload = {
        "model": model_name, "prompt": prompt, "stream": False, "keep_alive": "10m",
        "options": {"temperature": 0, "seed": 42, "num_predict": 160, "num_ctx": 2048}
    }
    response = requests.post(OLLAMA_URL + "/api/generate", json=payload, timeout=600)
    response.raise_for_status()
    result = response.json()
    count, duration = result.get("eval_count", 0), result.get("eval_duration", 0)
    speed = count / (duration / 1e9) if duration else 0
    return {
        "answer": result.get("response", "").strip(),
        "eval_count": count,
        "eval_duration_ns": duration,
        "tokens_per_second": speed,
        "prompt_eval_count": result.get("prompt_eval_count", 0),
        "prompt_eval_duration_ns": result.get("prompt_eval_duration", 0),
        "total_duration_ns": result.get("total_duration", 0)
    }


## 5. Run each setting on the same 10 prompts

In [ ]:
def benchmark(model_name, label):
    print(f"Loading and warming up {label}...")
    warmup = generate_one(model_name, "Reply with the word ready.")
    time.sleep(2)
    before = gpu_memory()
    rows = []
    for i, prompt in enumerate(prompts, 1):
        result = generate_one(model_name, prompt)
        rows.append({"setting": label, "model": model_name, "prompt_id": i, "prompt": prompt, **result})
        print(f"{label}: {i}/10 | {result['tokens_per_second']:.2f} tokens/sec")
    after = gpu_memory()
    return rows, {
        "setting": label, "model": model_name,
        "gpu_before": before, "gpu_after": after,
        "ollama_loaded_models": loaded_models(),
        "warmup_generated_tokens": warmup["eval_count"]
    }

results_q4, memory_q4 = benchmark(MODEL_Q4, "Q4_K_M")
results_q8, memory_q8 = benchmark(MODEL_Q8, "Q8_0")
all_results = results_q4 + results_q8
results_df = pd.DataFrame(all_results)
display(results_df[["setting", "prompt_id", "eval_count", "tokens_per_second"]])


Loading and warming up Q4_K_M...
Q4_K_M: 1/10 | 6.08 tokens/sec
Q4_K_M: 2/10 | 5.88 tokens/sec
Q4_K_M: 3/10 | 5.71 tokens/sec
Q4_K_M: 4/10 | 6.05 tokens/sec
Q4_K_M: 5/10 | 6.10 tokens/sec
Q4_K_M: 6/10 | 6.04 tokens/sec
Q4_K_M: 7/10 | 6.11 tokens/sec
Q4_K_M: 8/10 | 6.11 tokens/sec
Q4_K_M: 9/10 | 6.19 tokens/sec
Q4_K_M: 10/10 | 6.22 tokens/sec
Loading and warming up Q8_0...
Q8_0: 1/10 | 4.09 tokens/sec
Q8_0: 2/10 | 4.06 tokens/sec
Q8_0: 3/10 | 4.20 tokens/sec
Q8_0: 4/10 | 4.05 tokens/sec
Q8_0: 5/10 | 3.93 tokens/sec


## 6. Compare speed and memory

In [ ]:
summary_rows = []
for label, rows, memory in [
    ("Q4_K_M", results_q4, memory_q4),
    ("Q8_0", results_q8, memory_q8)
]:
    speeds = [r["tokens_per_second"] for r in rows]
    summary_rows.append({
        "setting": label,
        "mean_tokens_per_second": round(statistics.mean(speeds), 2),
        "median_tokens_per_second": round(statistics.median(speeds), 2),
        "min_tokens_per_second": round(min(speeds), 2),
        "max_tokens_per_second": round(max(speeds), 2),
        "gpu_memory_before_mb": memory["gpu_before"]["used_mb"],
        "gpu_memory_after_mb": memory["gpu_after"]["used_mb"]
    })

speed_memory_df = pd.DataFrame(summary_rows)
display(speed_memory_df)
print("Ollama loaded-model memory details:")
print(json.dumps({"Q4_K_M": memory_q4["ollama_loaded_models"], "Q8_0": memory_q8["ollama_loaded_models"]}, indent=2))


## 7. Compare answer quality

Review the two answers for each prompt. Rate each from 1 to 5: 1 = incorrect/unusable; 2 = mostly incorrect/incomplete; 3 = partly correct; 4 = correct/useful; 5 = correct, clear, and complete. Fill in the observations from the actual outputs.


In [ ]:
quality_df = results_df.pivot(index="prompt_id", columns="setting", values="answer").reset_index()
quality_df["Q4_rating_1_to_5"] = ""
quality_df["Q8_rating_1_to_5"] = ""
quality_df["observation"] = ""
display(quality_df)


## 8. Technical explanation: quantisation and KV cache

**Quantisation:** Quantisation stores model weights using fewer bits. 8-bit weights use less memory than 16-bit weights, and 4-bit weights use less memory than 8-bit weights. This can reduce memory requirements and improve deployment efficiency, but lower precision can introduce numerical error and may affect answer quality.

**KV cache:** During autoregressive generation, a transformer stores keys and values from previous tokens so they do not need to be recomputed at every generation step. This is the key-value (KV) cache. It grows with context length and batch size and consumes additional memory. Quantising model weights does not automatically remove KV-cache memory use.

Discuss from your measurements: which setting used less memory, which was faster, and whether answer quality differed. Explain that speed and memory depend on GPU offload, runtime, context length, prompt length, and concurrent load.


## 9. Back-of-the-envelope plan: 20 simultaneous users on one 24 GB GPU

State assumptions and estimate each item rather than assuming model weights are the entire memory budget.

| Component | Your estimate |
|---|---:|
| Model weights for selected quantisation | Use model/runtime information |
| KV cache per user | Estimate using model architecture, context length, and cache precision |
| KV cache for 20 users | Per-user estimate × 20 |
| Runtime, CUDA, temporary buffers | State an allowance |
| Total estimated VRAM | Sum the components |
| GPU capacity | 24 GB |
| Remaining headroom | 24 GB − total |

Discuss batching or request queuing, maximum context length, maximum output length, monitoring, and load testing. If the total exceeds 24 GB, say 20 simultaneous active generations are not feasible under those assumptions and propose queuing or reducing context/batch size. Do not invent KV-cache numbers without stating the model dimensions and assumptions.


## 10. Save results

In [ ]:
results_df.to_csv("task3_benchmark_results.csv", index=False)
speed_memory_df.to_csv("task3_speed_memory_summary.csv", index=False)

artifacts = {
    "model_q4": MODEL_Q4,
    "model_q8": MODEL_Q8,
    "prompts": prompts,
    "q4_memory": memory_q4,
    "q8_memory": memory_q8,
    "speed_memory_summary": summary_rows,
    "quality_review": quality_df.to_dict(orient="records"),
    "note": "Benchmark values are from this Colab runtime. Complete quality ratings and the 20-user serving estimate using observed outputs and stated assumptions."
}
with open("task3_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, indent=2, ensure_ascii=False)

print("Saved task3_benchmark_results.csv")
print("Saved task3_speed_memory_summary.csv")
print("Saved task3_benchmark_results.json")
